In [0]:
import pandas as pd
import numpy as np

feat = (spark.table("oil_shock.gold.fct_model_features_daily")
        .select("price_date", "brent", "brent_ret_1d", "brent_vol_20d").toPandas())
feat["price_date"] = pd.to_datetime(feat["price_date"])
feat = feat.sort_values("price_date").reset_index(drop=True)
dates = feat["price_date"].values
ret = feat["brent_ret_1d"].values
vol = feat["brent_vol_20d"].values
mu60 = pd.Series(ret).shift(1).rolling(60, min_periods=40).mean().values  # normal drift before the event
last_price_date = feat["price_date"].max()

WINDOWS = {"d0": (0, 0), "d0_1": (0, 1), "d0_3": (0, 3), "d0_5": (0, 5), "d6_20": (6, 20)}

def measure(event_date):
    i0 = int(np.searchsorted(dates, np.datetime64(pd.Timestamp(event_date))))
    out = {"day0_date": pd.Timestamp(dates[i0]) if i0 < len(dates) else pd.NaT}
    for name, (a, b) in WINDOWS.items():
        lo, hi = i0 + a, i0 + b
        k = hi - lo + 1
        if i0 >= len(dates) or hi >= len(dates) or np.isnan(vol[i0]) or np.isnan(mu60[i0]):
            out[f"move_pct_{name}"] = np.nan
            out[f"excess_pct_{name}"] = np.nan
            out[f"scar_{name}"] = np.nan
            continue
        car = ret[lo:hi + 1].sum()
        car_adj = car - mu60[i0] * k
        out[f"move_pct_{name}"] = 100 * (np.exp(car) - 1)        # what actually happened
        out[f"excess_pct_{name}"] = 100 * (np.exp(car_adj) - 1)  # beyond the normal trend
        out[f"scar_{name}"] = car_adj / (vol[i0] * np.sqrt(k))   # in units of normal volatility
    return out

# 1) Curated events (all of them)
cur = spark.table("oil_shock.silver.ref_event_timeline").toPandas()
cur["event_date"] = pd.to_datetime(cur["event_date"])
major_dates = cur.loc[cur["severity_1to5"] >= 4, "event_date"]
cur["overlaps_major_event"] = cur.apply(
    lambda r: bool(((major_dates - r["event_date"]).abs().dt.days.between(1, 7)).any()), axis=1)
cur = pd.concat([cur.reset_index(drop=True),
                 cur["event_date"].apply(lambda d: pd.Series(measure(d))).reset_index(drop=True)], axis=1)
cur["source"] = "curated"

# 2) Automatic events: Gulf surprise z >= 3, first day of each cluster
auto = spark.sql("""
    SELECT news_date AS event_date, oil_risk_zscore
    FROM oil_shock.gold.fct_risk_index_daily
    WHERE event_region = 'Gulf' AND NOT is_data_gap
      AND oil_risk_zscore >= 3 AND news_date >= '2025-04-01'
    ORDER BY news_date
""").toPandas()
auto["event_date"] = pd.to_datetime(auto["event_date"])
keep, last = [], None
for d in auto["event_date"]:
    if last is None or (d - last).days > 5:
        keep.append(d)
        last = d
auto = auto[auto["event_date"].isin(keep)].reset_index(drop=True)
auto["event"] = "Gulf risk surprise z = " + auto["oil_risk_zscore"].round(1).astype(str)
auto["category"] = "auto_gulf_surprise"
auto["region"] = "Gulf"
auto["severity_1to5"] = np.nan
auto["overlaps_major_event"] = False
auto = pd.concat([auto, auto["event_date"].apply(lambda d: pd.Series(measure(d)))], axis=1)
auto["source"] = "automatic"

# 3) Baseline: every trading day since April 2025
base = pd.DataFrame([measure(d) for d in feat.loc[feat["price_date"] >= "2025-04-01", "price_date"]])
pool = base["scar_d0_5"].dropna().abs().values
base_abs = pool.mean()

cols = (["source", "event_date", "day0_date", "category", "region", "severity_1to5", "event",
         "overlaps_major_event"] + [c for c in cur.columns if c.startswith(("move_pct_", "excess_pct_", "scar_"))])
events = pd.concat([cur[cols], auto[cols]], ignore_index=True)

def t_stat(x):
    x = x.dropna()
    return x.mean() / (x.std(ddof=1) / np.sqrt(len(x))) if len(x) > 2 else np.nan

rng = np.random.default_rng(42)
def boot_p(abs_values, B=5000):
    n = len(abs_values)
    if n == 0:
        return np.nan
    draws = rng.choice(pool, size=(B, n), replace=True).mean(axis=1)
    return (draws >= abs_values.mean()).mean()

measured = events.dropna(subset=["scar_d0_5"])

summary = (measured.groupby(["source", "category"])
           .agg(n=("scar_d0_5", "size"),
                n_clean=("overlaps_major_event", lambda x: int((~x.astype(bool)).sum())),
                avg_move_d0=("move_pct_d0", "mean"),
                avg_excess_d0_5=("excess_pct_d0_5", "mean"),
                avg_excess_d6_20=("excess_pct_d6_20", "mean"),
                pct_up_d0_5=("excess_pct_d0_5", lambda x: 100 * (x > 0).mean()),
                avg_abs_scar_d0_5=("scar_d0_5", lambda x: x.abs().mean()),
                t_signed_d0_5=("scar_d0_5", t_stat))
           .reset_index())
summary["abs_vs_normal"] = summary["avg_abs_scar_d0_5"] / base_abs
clean = (measured[~measured["overlaps_major_event"].astype(bool)]
         .groupby(["source", "category"])["excess_pct_d0_5"].mean().rename("avg_excess_d0_5_clean"))
summary = summary.merge(clean, on=["source", "category"], how="left")
summary = summary.round(2).sort_values(["source", "abs_vs_normal"], ascending=[True, False])

pooled_sets = {
    "curated: all": measured[measured["source"] == "curated"],
    "curated: severity 4-5": measured[(measured["source"] == "curated") & (measured["severity_1to5"] >= 4)],
    "curated: severity 4-5, no overlap": measured[(measured["source"] == "curated") & (measured["severity_1to5"] >= 4)
                                                  & (~measured["overlaps_major_event"].astype(bool))],
    "automatic: all": measured[measured["source"] == "automatic"],
}
pooled = pd.DataFrame([{
    "set": name,
    "n": len(df),
    "avg_abs_scar_d0_5": round(df["scar_d0_5"].abs().mean(), 2),
    "abs_vs_normal": round(df["scar_d0_5"].abs().mean() / base_abs, 2),
    "p_value_vs_random_days": round(boot_p(df["scar_d0_5"].abs().values), 4),
} for name, df in pooled_sets.items()])

print(f"Brent prices up to: {last_price_date.date()}")
print(f"Normal day (0-5 window) average size: {base_abs:.2f} | baseline days: {len(pool)}")
print(f"Events measured: {len(measured)} | waiting for prices: {events['scar_d0_5'].isna().sum()}")
display(pooled)
display(summary)

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# Save to Delta (ml schema)
for name, df in {"event_impact_events": events,
                 "event_impact_summary": summary,
                 "event_impact_pooled": pooled}.items():
    out = df.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

# Log to MLflow
mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-event-impact")
with mlflow.start_run(run_name=f"event_impact {run_stamp}"):
    mlflow.log_params({
        "price_series": "EIA Brent physical spot (RBRTE)",
        "vol_window_days": 20,
        "drift_window_days": 60,
        "windows": "d0, d0_1, d0_3, d0_5, d6_20",
        "auto_event_rule": "Gulf oil_risk_zscore >= 3, first day per 5-day cluster, from 2025-04-01",
        "overlap_rule": "another severity 4-5 event within 7 days",
        "bootstrap_draws": 5000,
        "brent_prices_up_to": str(last_price_date.date()),
    })
    mlflow.log_metrics({
        "baseline_abs_scar_d0_5": float(base_abs),
        "n_curated": int((measured["source"] == "curated").sum()),
        "n_automatic": int((measured["source"] == "automatic").sum()),
        "curated_abs_vs_normal": float(pooled.loc[pooled["set"] == "curated: all", "abs_vs_normal"].iloc[0]),
        "curated_p_value": float(pooled.loc[pooled["set"] == "curated: all", "p_value_vs_random_days"].iloc[0]),
        "major_abs_vs_normal": float(pooled.loc[pooled["set"] == "curated: severity 4-5", "abs_vs_normal"].iloc[0]),
        "automatic_abs_vs_normal": float(pooled.loc[pooled["set"] == "automatic: all", "abs_vs_normal"].iloc[0]),
        "automatic_p_value": float(pooled.loc[pooled["set"] == "automatic: all", "p_value_vs_random_days"].iloc[0]),
    })
    for name, df in {"events": events, "summary": summary, "pooled": pooled}.items():
        path = f"/tmp/event_impact_{name}.csv"
        df.to_csv(path, index=False)
        mlflow.log_artifact(path)
print("MLflow run logged.")